In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Contoh untuk MovieLens:
ratings = pd.read_csv('../data/raw/ratings.csv')
movies = pd.read_csv('../data/raw/movies.csv')

# Info dasar
print(ratings.shape, movies.shape)
print(ratings.head())
print(movies.info())

# Statistik rating
ratings['rating'].describe().T
sns.histplot(ratings['rating'], bins=10)

# Distribusi jumlah rating per film
movie_ratings_count = ratings.groupby('movieId').size()
sns.histplot(movie_ratings_count, bins=50)
plt.title('Jumlah Rating per Film')

# Film terpopuler
top_movies = movie_ratings_count.sort_values(ascending=False).head(10)
# Gabung dengan judul
top_movies_with_title = movies[movies['movieId'].isin(top_movies.index)][['movieId','title']]
top_movies_with_title['count'] = top_movies.values
print(top_movies_with_title)

A. Input (Fitur yang Digunakan)
    Sistem rekomendasi film ini akan memanfaatkan kombinasi data interaksi pengguna dan atribut item, yang terbagi menjadi beberapa opsi pendekatan:
        - Data Interaksi (Collaborative Filtering):
            a. userId: Identitas unik pengguna.
            b. movieId: Identitas unik film.
            c. rating: Nilai eksplisit (1.0 - 5.0) yang diberikan oleh user kepada movie (riwayat interaksi sebagai sinyal preferensi).
        - Data Konten / Metadata (Content-Based Filtering):
            a. title: Judul film (dapat diekstraksi tahun rilisnya sebagai fitur tambahan).
            b. genres: Kategori film (misal: Action|Sci-Fi|Thriller) yang diubah menjadi representasi One-Hot Encoding atau TF-IDF / Word Embeddings.

B. Output (Target yang Diprediksi/Direkomendasikan)
    Berdasarkan kebutuhan sistem, output proyek ini dipetakan menjadi dua bentuk:
        - Prediksi Rating (Rating Prediction): Mengeluarkan nilai estimasi numerik kontinyu (skala 1.0 sampai 5.0) terhadap seberapa besar seorang user tertentu akan menyukai movie yang belum pernah ia tonton.
        - Top-N Recommendations (Ranking): Menghasilkan daftar urutan terbaik berisi $N$ film (misalnya Top-10 Film Rekomendasi) yang paling relevan dan dipersonalisasi untuk setiap user, atau daftar film yang paling mirip dengan suatu film acuan (Item-to-Item similarity).

C. Pendekatan (Methodology & Hypothesis)
    Proyek ini menggunakan pendekatan Supervised Learning (untuk prediksi eksplisit) dan Unsupervised/Ranking-based (untuk kemiripan item). Kita akan menguji dan membandingkan 3 hipotesis pemodelan:
        - Pendekatan Baseline — Collaborative Filtering (Matrix Factorization / SVD):
            a. Metode: Menggunakan algoritma Singular Value Decomposition (SVD) atau Alternating Least Squares (ALS) untuk memecahkan matriks User-Item Rating menjadi latent factor vectors.
            b. Hipotesis: Pengguna yang memiliki selera rating serupa di masa lalu akan memiliki kecenderungan menyukai film yang sama di masa depan tanpa perlu mengetahui genre atau isi cerita filmnya
        - Pendekatan Deep Learning — Neural Collaborative Filtering (NCF):
            a. Metode: Menggabungkan Generalized Matrix Factorization (GMF) dan Multi-Layer Perceptron (MLP) dalam satu arsitektur Neural Network untuk mempelajari interaksi non-linear antara user embedding dan item embedding.
            b. Hipotesis: Arsitektur Deep Learning mampu menangkap pola interaksi yang lebih kompleks dan non-linear dibandingkan perkalian matriks linear biasa pada SVD konvensional.
        - Pendekatan Alternatif — Content-Based Filtering (Cosine Similarity):
            a. Metode: Menghitung vektor kemiripan antar film berdasarkan fitur genres atau metadata lainnya menggunakan TF-IDF Vectorizer dan Cosine Similarity.
            b. Hipotesis: Sangat efektif sebagai fallback strategy untuk mengatasi masalah Cold-Start (ketika ada film baru yang belum memiliki history rating dari user sama sekali).

D. Metrik Sukses (Evaluation Metrics)
    Evaluasi model akan dilakukan dalam dua tahap sesuai dengan tipe output yang dihasilkan:
        - Untuk Evaluasi Prediksi Rating (Error Metrics):RMSE (Root Mean Squared Error): Menilai    seberapa jauh melesetnya angka rating prediksi model terhadap rating asli yang diberikan user. Metrik ini memberikan penalti lebih besar pada kesalahan prediksi yang ekstrem. MAE (Mean Absolute Error): Menilai rata-rata kesalahan mutlak prediksi rating.

        - Untuk Evaluasi Rekomendasi Ranking (Information Retrieval Metrics):Precision@K (misal: Precision@10): Mengukur berapa persentase film di dalam daftar 10 rekomendasi teratas yang benar-benar disukai atau relevan bagi pengguna. Recall@K: Mengukur seberapa banyak film yang disukai pengguna berhasil ditangkap ke dalam daftar 10 rekomendasi teratas. NDCG@K (Normalized Discounted Cumulative Gain): Mengevaluasi kualitas urutan/ranking rekomendasi (film yang paling relevan harus berada di urutan paling atas).